# 02 — POWSM Inference

Run Phone Recognition (PR) and Grapheme-to-Phoneme (G2P) on all preprocessed WAV files.
Results are saved to `results/pr/` and `results/g2p/`.

In [ ]:
import sys, json
from pathlib import Path
from datetime import datetime
import numpy as np
import soundfile as sf

sys.path.insert(0, str(Path("..").resolve()))

from config import DATA_DIR, RESULTS_DIR, DEVICE, LANG, MODEL_NAME
from powsm.espnet_subsampling_prefix_compat import apply_subsampling_prefix_compat
from powsm.ipa import cmu_target_ipa, tokenize_ipa

In [ ]:
apply_subsampling_prefix_compat()

from espnet2.bin.s2t_inference_ctc import Speech2TextGreedySearch

model = Speech2TextGreedySearch.from_pretrained(
    MODEL_NAME,
    device=DEVICE,
    lang_sym=LANG,
    task_sym="<pr>",
)
print("Model loaded.")

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Model loaded.


In [ ]:
def parse_pr(raw: str) -> list[str]:
    """Parse POWSM PR output into a list of phone tokens."""
    if "<notimestamps>" in raw:
        raw = raw.split("<notimestamps>")[1]
    return [p.strip().strip("/") for p in raw.strip().split("//") if p.strip().strip("/")]

for sentence_dir in sorted(DATA_DIR.iterdir()):
    sid = sentence_dir.name
    text_file = sentence_dir / "text"
    transcript = text_file.read_text().strip() if text_file.exists() else ""

    # G2P via CMU dictionary (canonical target pronunciation)
    target_ipa = cmu_target_ipa(transcript)
    target_phones = tokenize_ipa(target_ipa)

    print(f"\n=== Sentence {sid} ===")
    print(f"Transcript: {transcript}")
    print(f"G2P (CMU): {len(target_phones)} phones")

    g2p_result = {
        "sentence_id": sid,
        "transcript": transcript,
        "model": "cmu_dict",
        "g2p": {"raw": target_ipa, "phones": target_phones},
    }
    g2p_dir = RESULTS_DIR / "g2p" / sid
    g2p_dir.mkdir(parents=True, exist_ok=True)
    (g2p_dir / "target.json").write_text(json.dumps(g2p_result, ensure_ascii=False, indent=2))

    for wav_file in sorted(sentence_dir.glob("*.wav")):
        speech, rate = sf.read(str(wav_file))
        audio = np.squeeze(speech).astype(np.float32)
        speaker = wav_file.stem
        print(f"\n  Speaker: {speaker}")

        # PR via powsm-ctc
        raw = model.batch_decode([audio], batch_size=1)[0]
        phones = parse_pr(raw)
        print(f"  PR: {len(phones)} phones — {' '.join(phones[:10])}...")

        pr_result = {
            "sentence_id": sid,
            "speaker": speaker,
            "audio_file": wav_file.name,
            "transcript": transcript,
            "timestamp": datetime.now().isoformat(),
            "model": MODEL_NAME,
            "phone_recognition": {"raw": raw, "phones": phones},
        }
        pr_dir = RESULTS_DIR / "pr" / sid
        pr_dir.mkdir(parents=True, exist_ok=True)
        (pr_dir / f"{speaker}.json").write_text(json.dumps(pr_result, ensure_ascii=False, indent=2))

print("\nDone. Results saved to", RESULTS_DIR)


=== Sentence 12 ===
Transcript: The weather is rather warm this Thursday. I think we should go to the theater together. Thank you for thinking about this thoroughly.
G2P (CMU): 78 phones

  Speaker: umit12
  PR: 80 phones — t u w e d ə r ɨ d r...

  Speaker: yusuf12
  PR: 79 phones — t u b ɛ d ɛ r r ɛ d...

=== Sentence 14 ===
Transcript: The red car arrived early in the morning. The driver parked near the restaurant and ordered breakfast. The fresh bread was really delicious.
G2P (CMU): 90 phones

  Speaker: umit14
  PR: 90 phones — d ə r ɛ d k a r ɛ r...

Done. Results saved to C:\Users\faruq\Desktop\college\senior\sig\exp-faruq\results
